In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Remove this line if you want interactive plots

df = pd.read_csv("fruits.csv", parse_dates=["date"])

print("=== Head ===")
print(df.head())

print("\n=== Describe ===")
print(df.describe())

=== Head ===
    fruit  price  quantity       date
0   Apple   1.20        50 2025-01-01
1  Banana   0.50       120 2025-01-01
2  Orange   0.80        75 2025-01-01
3   Mango   2.50        30 2025-01-01
4   Apple   1.25        45 2025-02-01

=== Describe ===
           price    quantity                 date
count  16.000000   16.000000                   16
mean    1.264375   68.562500  2025-02-15 00:00:00
min     0.480000   25.000000  2025-01-01 00:00:00
25%     0.722500   38.750000  2025-01-24 06:00:00
50%     1.025000   62.500000  2025-02-15 00:00:00
75%     1.575000   91.250000  2025-03-08 18:00:00
max     2.600000  130.000000  2025-04-01 00:00:00
std     0.776462   35.105496                  NaN


In [2]:
grouped = df.groupby("fruit").agg(
    avg_price=("price", "mean"),
    total_quantity=("quantity", "sum")
).reset_index()

print("=== Group By Fruit ===")
print(grouped)

=== Group By Fruit ===
    fruit  avg_price  total_quantity
0   Apple     1.2250             190
1  Banana     0.5125             475
2   Mango     2.4875             122
3  Orange     0.8325             310


In [3]:
filtered = df[df["price"] > 1.0].sort_values("price", ascending=False)

print("=== Fruits with price > 1.00 (sorted desc) ===")
print(filtered)

=== Fruits with price > 1.00 (sorted desc) ===
    fruit  price  quantity       date
11  Mango   2.60        25 2025-03-01
3   Mango   2.50        30 2025-01-01
15  Mango   2.45        32 2025-04-01
7   Mango   2.40        35 2025-02-01
8   Apple   1.30        40 2025-03-01
4   Apple   1.25        45 2025-02-01
0   Apple   1.20        50 2025-01-01
12  Apple   1.15        55 2025-04-01


In [4]:
fig, ax = plt.subplots(figsize=(10, 6))

for fruit in df["fruit"].unique():
    fruit_data = df[df["fruit"] == fruit]
    ax.plot(fruit_data["date"], fruit_data["price"], marker="o", label=fruit)

ax.set_title("Fruit Prices Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Price ($)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fruit_prices.png", dpi=150)
plt.show()
print("Plot saved to fruit_prices.png")

Plot saved to fruit_prices.png


In [5]:
import polars as pl
import time

# --- Pandas ---
start = time.perf_counter()
pd_result = df.groupby("fruit").agg(
    avg_price=("price", "mean"),
    total_quantity=("quantity", "sum")
)
pandas_time = time.perf_counter() - start

# --- Polars ---
df_pl = pl.read_csv("fruits.csv")

start = time.perf_counter()
pl_result = df_pl.group_by("fruit").agg(
    pl.col("price").mean().alias("avg_price"),
    pl.col("quantity").sum().alias("total_quantity"),
)
polars_time = time.perf_counter() - start

print(f"Pandas time:  {pandas_time:.6f}s")
print(f"Polars time:  {polars_time:.6f}s")
print(f"Speedup:      {pandas_time / polars_time:.2f}x")

print("\n=== Polars Result ===")
print(pl_result)

Pandas time:  0.004251s
Polars time:  0.107245s
Speedup:      0.04x

=== Polars Result ===
shape: (4, 3)
┌────────┬───────────┬────────────────┐
│ fruit  ┆ avg_price ┆ total_quantity │
│ ---    ┆ ---       ┆ ---            │
│ str    ┆ f64       ┆ i64            │
╞════════╪═══════════╪════════════════╡
│ Orange ┆ 0.8325    ┆ 310            │
│ Mango  ┆ 2.4875    ┆ 122            │
│ Apple  ┆ 1.225     ┆ 190            │
│ Banana ┆ 0.5125    ┆ 475            │
└────────┴───────────┴────────────────┘


In [7]:
pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 74.2 MB/s  0:00:00s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn]0m 3/4 [scikit-learn]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
from sklearn.datasets import load_iris
import polars as pl
import matplotlib.pyplot as plt

# Load Iris
iris = load_iris()
df_iris = pl.DataFrame({
    "sepal_length": iris.data[:, 0],
    "sepal_width": iris.data[:, 1],
    "petal_length": iris.data[:, 2],
    "petal_width": iris.data[:, 3],
    "species": [iris.target_names[t] for t in iris.target],
})

print(df_iris.head())
print(df_iris.describe())

# Complex filter: large petals on non-setosa species
filtered = df_iris.filter(
    (pl.col("species") != "setosa") &
    (pl.col("petal_length") > 4.5)
).sort(["species", "petal_length"], descending=[False, True])

print("\n=== Filtered & Sorted ===")
print(filtered)

# Scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
for species in df_iris["species"].unique().to_list():
    subset = df_iris.filter(pl.col("species") == species)
    ax.scatter(subset["sepal_length"], subset["petal_length"], label=species, alpha=0.7)

ax.set_xlabel("Sepal Length")
ax.set_ylabel("Petal Length")
ax.set_title("Iris: Sepal vs Petal Length")
ax.legend()
plt.tight_layout()
plt.savefig("iris_scatter.png", dpi=150)
plt.show()

shape: (5, 5)
┌──────────────┬─────────────┬──────────────┬─────────────┬─────────┐
│ sepal_length ┆ sepal_width ┆ petal_length ┆ petal_width ┆ species │
│ ---          ┆ ---         ┆ ---          ┆ ---         ┆ ---     │
│ f64          ┆ f64         ┆ f64          ┆ f64         ┆ str     │
╞══════════════╪═════════════╪══════════════╪═════════════╪═════════╡
│ 5.1          ┆ 3.5         ┆ 1.4          ┆ 0.2         ┆ setosa  │
│ 4.9          ┆ 3.0         ┆ 1.4          ┆ 0.2         ┆ setosa  │
│ 4.7          ┆ 3.2         ┆ 1.3          ┆ 0.2         ┆ setosa  │
│ 4.6          ┆ 3.1         ┆ 1.5          ┆ 0.2         ┆ setosa  │
│ 5.0          ┆ 3.6         ┆ 1.4          ┆ 0.2         ┆ setosa  │
└──────────────┴─────────────┴──────────────┴─────────────┴─────────┘
shape: (9, 6)
┌────────────┬──────────────┬─────────────┬──────────────┬─────────────┬───────────┐
│ statistic  ┆ sepal_length ┆ sepal_width ┆ petal_length ┆ petal_width ┆ species   │
│ ---        ┆ ---          ┆ --